In [1]:
import os
import sys
import tempfile
import pandas as pd
import numpy as np
import json 
import torch

import matplotlib.pyplot as plt

from datetime import datetime

from numpy.random import default_rng

# import packages from the project with sys
sys.path.append("/workspace/meteolibre_model")

In [2]:
from meteolibre_model.datasets.dataset_meteofrance_v2 import (
    MeteoLibreDataset,
)
from torch.utils.data import DataLoader

PATHDATA = ["/workspace/data/hf_dataset_v0/", "/workspace/data/hf_dataset_v1/"]
dataset = MeteoLibreDataset(directory=PATHDATA)




In [8]:
element = []

for value in dataset.index_data["time_radar"].values:
    if value[0] != 2. or  value[-1] != -2.:
        element.append(True)
    else:
        element.append(False)
    

In [9]:
sum(element)

12962

In [ ]:
torch.manual_seed(48)

train_dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True,
    num_workers=6,
    
)  # 

In [ ]:


def calculate_global_stats(loader, key="satellite", batch_size=16):
    """
    Calculates the mean and standard deviation per channel for a dataset.
    """
    channels_sum_satellite, channels_squared_sum_satellite, num_batches = 0, 0, 0

    i_max = 100 

    # The '.dataset' is important if the loader wraps a Subset
    num_pixels = i_max * batch_size * 256 * 256 * 9 # Replace with your image dimensions

    i = 0

    for data in loader:
        # data shape is [B, C, H, W]
        # Sum over all axes except the channel axis
        #print(data[key].shape)
        channels_sum_satellite += torch.sum(data[key], dim=[0, 1, 3, 4])
        channels_squared_sum_satellite += torch.sum(data[key]**2, dim=[0, 1, 3, 4])
        num_batches += 1

        i = i + 1
        
        if i > i_max:
            break

    # Calculate mean and std
    mean = channels_sum_satellite / num_pixels
    # E[X^2] - (E[X])^2
    std = torch.sqrt((channels_squared_sum_satellite / num_pixels) - mean**2)

    return mean, std



dataset_mean, dataset_std = calculate_global_stats(train_dataloader)
print()
print(f"Calculated Mean: {dataset_mean}")
print(f"Calculated Std: {dataset_std}")

In [ ]:
for batch in train_dataloader:
    break


In [ ]:
from meteolibre_model.dit.pl_model_meteofrance_simplediffusion import (
    Simple3DDiffusion,
)

model = Simple3DDiffusion(test_dataloader=train_dataloader,
        dir_save="./",
        parametrization="velocity",
        schedule="cosine",)

x_image, x_image_corrupt, mask_radar, mask_groundstation = model.prepare_target(
            batch
        )

In [ ]:
x_image = x_image.repeat(16, 1, 1, 1, 1)


In [ ]:


batch_size = x_image.shape[0]

target_meteo_frames = x_image[:, model.nb_back : (model.nb_back + model.nb_future)]
input_meteo_frames = x_image[:, : model.nb_back]

# Prior sample (simple Gaussian noise) - you can refine this prior
prior_image = model.init_prior(target_meteo_frames.shape)

# Time variable for Rectified Flow - sample uniformly
t = (
    torch.rand(batch_size, 1, 1, 1, 1)
    .type_as(target_meteo_frames)
    .to(target_meteo_frames.device)
)  # (B, 1)

t, _ = torch.sort(t, dim=0)


# we create a scalar value to condition the model on time stamp
# and hours
x_hour = batch["hour"].clone().detach().float().unsqueeze(1).repeat(16, 1)  # (B, 1)
x_minute = batch["minute"].clone().detach().float().unsqueeze(1).repeat(16, 1)  # (B, 1)

# Simple scalar condition: hour of the day and minutes of the day. You might want to expand this.
x_scalar = torch.cat([x_hour, x_minute, t[:, :, 0, 0, 0]], dim=1)

# Interpolate between prior and data to get x_t
schedule, schedule_deriv = model.get_proper_schedule(t)

x_t = schedule * target_meteo_frames + (1 - schedule) * prior_image



In [ ]:
toplot = x_t[:, 0, -2, :, :]

plt.figure()
plt.imshow(target_meteo_frames[0, 0, -3, :, :])
plt.colorbar()

for i in range(toplot.shape[0]):
    plt.figure()
    print(schedule[i])
    plt.imshow(toplot[i])
    plt.colorbar()


In [ ]:
toplot = x_t[:, 0, -2, :, :]

plt.figure()
plt.imshow(target_meteo_frames[0, 0, -3, :, :])
plt.colorbar()

for i in range(toplot.shape[0]):
    plt.figure()
    print(schedule[i])
    plt.imshow(toplot[i])
    plt.colorbar()


In [ ]:
toplot.shape


In [ ]:
data = dataset[6000]
print(data.keys())

In [ ]:
dataset.index_data.iloc[6000]

In [ ]:
sattelite = data['satellite']

for i in range(sattelite.shape[0]):
    # we plot the sattelite and the sattelite
    plt.figure(figsize=(10, 5))
    plt.imshow(sattelite[i, -3, :, :], cmap='gray')
    plt.title('ground height')
    plt.colorbar()


In [ ]:
sattelite